In [1]:
import torch
import numpy as np
import pandas as pd
import json
import os
from tqdm import tqdm

# 导入您项目中的工具函数和模块
import trainUtils
import ioutils

# 导入ESM相关的库
from esm.sdk.api import ESMProtein
from esm.utils.constants import esm3 as C
from esm.tokenization.sequence_tokenizer import EsmSequenceTokenizer

# --- 1. 加载您训练好的模型 ---
print("--- 正在加载模型 ---")
# 这里我们假设加载的是您微调后的模型
CONFIG_PATH = "/data2/zhoukaitao/01evoModel/checkpoints/250904_local_7countries/config.json"
LORA_CHECKPOINT_PATH = "/data2/zhoukaitao/01evoModel/checkpoints/250904_local_7countries/epoch=54-validation_loss=0.5596.ckpt"
DEVICE = "cuda:3" if torch.cuda.is_available() else "cpu"

with open(CONFIG_PATH, 'r') as f:
    configs_temp = json.load(f)

local_pretrain_model = trainUtils.loadPretrainModel(configs_temp)
model = trainUtils.buildModel(configs_temp, local_pretrain_model, LORA_CHECKPOINT_PATH)
model.eval()
model.to(DEVICE)
print("模型加载完成。")

# --- 2. 加载分词器 ---
tokenizer = EsmSequenceTokenizer()
print("序列分词器加载完成。")

/home/zhoukaitao/anaconda3/envs/esm3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- 正在加载模型 ---
load local model: /data2/zhoukaitao/01evoModel/checkpoints/250813_stage1_test/pretrain_stage1_250824.pth
model at stage: training stage 1


/home/zhoukaitao/anaconda3/envs/esm3/lib/python3.10/site-packages/torch/nn/init.py:452: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


load model from checkpoint /data2/zhoukaitao/01evoModel/checkpoints/250904_local_7countries/epoch=54-validation_loss=0.5596.ckpt
模型加载完成。
序列分词器加载完成。


In [3]:
print("--- 正在准备结构约束数据 ---")

# --- 1. 从PDB文件加载蛋白质 ---
# PDB文件将作为我们所有信息的唯一来源
try:
    protein_with_structure = ESMProtein.from_pdb("pdb/S.pdb")
    print(f"已从PDB文件加载蛋白质，序列长度: {len(protein_with_structure.sequence)}")
except FileNotFoundError:
    raise FileNotFoundError("错误: 找不到 pdb/S.pdb 文件！请确保文件存在。")

# --- 2. 使用 model.encode() 获取对齐的结构Token ---
print("正在使用 model.encode() 获取 structure_t...")
with torch.no_grad():
    encoded_protein = local_pretrain_model.encode(protein_with_structure)
    # 从编码结果中提取结构Token
    original_structure_tokens = encoded_protein.structure.to(DEVICE)

print(f"结构Token获取成功，形状为: {original_structure_tokens.shape}")

# --- 3. 准备好用于采样的固定结构输入 ---
# 我们为它增加一个批次维度，以便在循环中直接使用
structure_t_input = original_structure_tokens.unsqueeze(0)

--- 正在准备结构约束数据 ---
已从PDB文件加载蛋白质，序列长度: 1273
正在使用 model.encode() 获取 structure_t...
结构Token获取成功，形状为: torch.Size([1275])


In [4]:
def gibbs_sample_with_structure_tokens(
    model, 
    tokenizer,
    structure_t: torch.Tensor,
    iterations: int, 
    temperature: float = 1.0, 
    protein_name: str = "S"
):
    """
    在离散化的结构Token约束下，使用吉布斯采样方法生成蛋白质序列。
    """
    device = model.device
    # 从结构Token的长度推断出序列长度
    sequence_length = structure_t.shape[1] - 2
    print(f"\n开始带结构Token约束的吉布斯采样，长度: {sequence_length}, 迭代次数: {iterations}")

    # --- 1. 初始化 ---
    bos_token, eos_token, mask_token = C.SEQUENCE_BOS_TOKEN, C.SEQUENCE_EOS_TOKEN, C.SEQUENCE_MASK_TOKEN
    
    # a. 创建一个完全由 <mask> 组成的初始序列
    sequence_tokens = torch.full((1, sequence_length + 2), mask_token, dtype=torch.long, device=device)
    sequence_tokens[0, 0] = bos_token
    sequence_tokens[0, -1] = eos_token
    
    pbar = tqdm(range(iterations), desc="Gibbs Sampling Iterations")

    # --- 2. 迭代采样主循环 ---
    with torch.no_grad():
        for step in pbar:
            position_to_update = torch.randint(1, sequence_length + 1, (1,)).item()

            # --- 核心修改：在输入字典中加入固定的 structure_t ---
            input_dict = {
                protein_name: {
                    'seq_t': sequence_tokens,
                    'structure_t': structure_t
                }
            }
            
            outputs = model(input_dict)
            logits = outputs.S1Logits[protein_name]
            position_logits = logits[0, position_to_update, :]
            
            position_logits /= temperature
            probabilities = torch.softmax(position_logits, dim=-1)
            new_token_id = torch.multinomial(probabilities, num_samples=1)
            sequence_tokens[0, position_to_update] = new_token_id

    # --- 3. 解码最终序列 ---
    final_token_ids = sequence_tokens.squeeze(0).cpu().numpy()
    final_sequence = tokenizer.decode(final_token_ids[1:-1])

    return final_sequence

In [5]:
# --- 设置生成参数 ---
# 长度由PDB文件决定
SEQ_LENGTH_FROM_PDB = len(protein_with_structure.sequence)
# 迭代次数越多，序列质量通常越好
ITERATIONS = SEQ_LENGTH_FROM_PDB * 5
TEMPERATURE = 1.0

# --- 运行采样 ---
generated_sequence = gibbs_sample_with_structure_tokens(
    model=model,
    tokenizer=tokenizer,
    structure_t=structure_t_input,
    iterations=ITERATIONS,
    temperature=TEMPERATURE,
    protein_name="S"
)

# --- 打印结果 ---
print("\n" + "="*50)
print("      带结构Token约束的吉布斯采样生成结果")
print("="*50)
print(f"生成的序列长度: {len(generated_sequence)}")
print("\n生成的序列 (前100个氨基酸):")
print(generated_sequence[:100])
print("\n生成的序列 (后100个氨基酸):")
print(generated_sequence[-100:])


开始带结构Token约束的吉布斯采样，长度: 1273, 迭代次数: 6365


Gibbs Sampling Iterations: 100%|██████████| 6365/6365 [21:43<00:00,  4.88it/s]


      带结构Token约束的吉布斯采样生成结果
生成的序列长度: 2580

生成的序列 (前100个氨基酸):
M F V F L V L L P L V S S Q C V N L I T R T Q - - - S Y T N S F T R G V Y Y P D K V F R S S V L H S 

生成的序列 (后100个氨基酸):
 L I A I V M V T I M L C C M T S C C S C L K I C C S C G S C C K F D E D D S E P V L K G V K L H Y T


In [6]:
generated_sequence

'M F V F L V L L P L V S S Q C V N L I T R T Q - - - S Y T N S F T R G V Y Y P D K V F R S S V L H S T Q D L F L P F F S N V T W F H A I - - S G T N G T K R F D N P V L P F N D G V Y F A S T E K S N I I R I W I F G T T L D S K T Q S L L I V N N A T N V V I K V C E F Q F C N D P F L D V Y Y H K N N K S W M E S E F R V Y S S A N N C T F E Y V S Q P F L M D L E G K Q G N F K N L R E F V F K N I D G Y F K I Y S K H T P I N L G R D L P Q G F S A L E P L V D L P I G I N I T R F Q T L L R L H R S Y L T P G D S S S G W T A G E A A Y Y V G Y L Q P R T F L L K Y N E N G T I T D A V D C A L D P L S E G K C T L K S F T V E K G I Y Q T S N F R V Q P T E S I V R F P N T T N L C P F D E V F N A T R F A <mask> V Y A W N R K R I S N C V A D Y S V L Y N F A P F F A F K C Y G V S P T K L N D L C F T N V Y A D S N V I R G N E V S Q I A P G Q T G N I A D Y N Y K L P D D F T G C V I A W N S N K L D S T V G G N Y N Y R Y R L F R K S K L K P F E R D I S T E I Y Q A G N K P C N G V A G V N C Y F P L Q S Y G F 